# EffDL-26: Liger-Kernel report

### Part 1. Usage for LLM training and benchmarks

In [ ]:
!./run_train_benchmark.sh
!python3 plots_utils.py

![img](imgs/perf_real_x_axis.png)

![img2](imgs/perf_uniform_x_axis.png)

### Part 2. The miracle behind LigerCrossEntropyLoss

In [1]:
import torch
import torch.nn.functional as F
from liger_kernel.transformers import LigerCrossEntropyLoss
from torch.nn import CrossEntropyLoss

VOCAB_SIZE = 32_768
SEQ_LEN = 1024

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
assert device == torch.device("cuda")


@torch.autograd.profiler.record_function("dummy_func")
def dummy_func():
    a = torch.ones(4096, 4096)
    a = a.to(device)
    a.sum().cpu()
    del a


def cross_entropy_memory_consumption(seq_len, vocab_size, loss_fn, run_backward = True):
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    
    logits = torch.randn(seq_len, vocab_size, device=device, requires_grad=True)
    targets = torch.randint(low=0, high=vocab_size, size=(seq_len,), device=device)
    
    loss = loss_fn(logits, targets)
    if run_backward:
        loss.backward()

    torch.cuda.synchronize()
    return torch.cuda.max_memory_allocated()


def cross_entropy_memory_snapshot(seq_len, vocab_size, loss_fn, run_backward = True):
    snapshot_name = loss_fn.__repr__()[:-2] + ("ForwardBackward" if run_backward else "Forward")
    torch.cuda.empty_cache()
    torch.cuda.memory._record_memory_history()
    
    logits = torch.randn(seq_len, vocab_size, device=device, requires_grad=True)
    targets = torch.randint(low=0, high=vocab_size, size=(seq_len,), device=device)
    
    loss = loss_fn(logits, targets)
    if run_backward:
        loss.backward()
    
    torch.cuda.synchronize()
    torch.cuda.memory._dump_snapshot(f"./snapshots/{snapshot_name}.pickle")
    

def cross_entropy_profile(seq_len, vocab_size, loss_fn):
    torch.cuda.empty_cache()
    with torch.profiler.profile(
        activities=[torch.profiler.ProfilerActivity.CPU, torch.profiler.ProfilerActivity.CUDA],
        schedule=torch.profiler.schedule(wait=3, warmup=3, active=1),
        record_shapes=True
    ) as prof:
        for _ in range(7):
            logits = torch.randn(seq_len, vocab_size, device=device, requires_grad=True)
            targets = torch.randint(low=0, high=vocab_size, size=(seq_len,), device=device)
            
            loss = loss_fn(logits, targets)
            dummy_func()
            loss.backward()

            prof.step()
        
    torch.cuda.synchronize()
    prof.export_chrome_trace(f"./traces/{loss_fn.__repr__()[:-2]}.json")


def cross_entropy_tensor_state(seq_len, vocab_size, loss_fn, run_backward = True):
    logits_cpu = torch.randn(seq_len, vocab_size)
    targets_cpu = torch.randint(low=0, high=vocab_size, size=(seq_len,))

    logits_cuda = logits_cpu.detach().to(device).requires_grad_(True)
    targets_cuda = targets_cpu.detach().to(device)
    
    loss = loss_fn(logits_cuda, targets_cuda)
    if run_backward:
        loss.backward()
        
    torch.cuda.synchronize()
    return logits_cpu, logits_cuda, targets_cpu, targets_cuda


#### Memory snapshots

In [5]:
!mkdir -p snapshots

In [6]:
torch_ce_loss = CrossEntropyLoss().to(device)
cross_entropy_memory_snapshot(SEQ_LEN, VOCAB_SIZE, torch_ce_loss, run_backward=False) 

In [2]:
torch_ce_loss = CrossEntropyLoss().to(device)
cross_entropy_memory_snapshot(SEQ_LEN, VOCAB_SIZE, torch_ce_loss, run_backward=True) 

In [2]:
liger_ce_loss = LigerCrossEntropyLoss().to(device)
cross_entropy_memory_snapshot(SEQ_LEN, VOCAB_SIZE, liger_ce_loss, run_backward=False) 

In [2]:
liger_ce_loss = LigerCrossEntropyLoss().to(device)
cross_entropy_memory_snapshot(SEQ_LEN, VOCAB_SIZE, liger_ce_loss, run_backward=True) 

#### Tensor state

In [2]:
# Torch case
for run_backward in [False, True]:
    torch_ce_loss = CrossEntropyLoss().to(device)
    logits_cpu, logits_cuda, targets_cpu, targets_cuda = cross_entropy_tensor_state(SEQ_LEN, VOCAB_SIZE, torch_ce_loss, run_backward=run_backward) 

    if not run_backward:
        assert logits_cpu.grad is None and logits_cuda.grad is None
    else:
        assert logits_cpu.grad is None and logits_cuda.grad is not None
        
    assert torch.allclose(targets_cpu, targets_cuda.cpu())
    assert torch.allclose(logits_cpu, logits_cuda.cpu())
    

In [3]:
# Liger-Kernel case
for run_backward in [False, True]:
    liger_ce_loss = LigerCrossEntropyLoss().to(device)
    logits_cpu, logits_cuda, targets_cpu, targets_cuda = cross_entropy_tensor_state(SEQ_LEN, VOCAB_SIZE, liger_ce_loss, run_backward=run_backward) 

    if not run_backward:
        assert logits_cpu.grad is None and logits_cuda.grad is None
    else:
        assert logits_cpu.grad is None and logits_cuda.grad is not None
        
    assert torch.allclose(targets_cpu, targets_cuda.cpu())
    assert torch.allclose(logits_cpu, logits_cuda.cpu()), f"Forward{" + backward" if run_backward else ""} mode"

AssertionError: Forward mode

In [4]:
# Liger-Kernel case
for run_backward in [False, True]:
    liger_ce_loss = LigerCrossEntropyLoss().to(device)
    logits_cpu, logits_cuda, targets_cpu, targets_cuda = cross_entropy_tensor_state(SEQ_LEN, VOCAB_SIZE, liger_ce_loss, run_backward=run_backward)

    if not run_backward:
        assert logits_cpu.grad is None and logits_cuda.grad is None
    else:
        assert logits_cpu.grad is None and logits_cuda.grad is not None
        
    print(torch.allclose(targets_cpu, targets_cuda.cpu()))
    print(f"Forward{" + backward" if run_backward else ""} mode:", torch.allclose(logits_cpu, logits_cuda.cpu()))

True
Forward mode: False
True
Forward + backward mode: False


In [5]:
liger_ce_loss = LigerCrossEntropyLoss().to(device)
logits_cpu, logits_cuda, targets_cpu, targets_cuda = cross_entropy_tensor_state(SEQ_LEN, VOCAB_SIZE, liger_ce_loss, run_backward=True)

targets_one_hot = F.one_hot(targets_cuda, num_classes=VOCAB_SIZE).to(dtype=logits_cuda.dtype)

print(logits_cuda.data_ptr(), logits_cuda.grad.data_ptr())
print(torch.allclose(logits_cuda.data, (F.softmax(logits_cpu.to(device), dim=-1) - targets_one_hot) / SEQ_LEN))

139868294348800 139868294348800
True


##### CrossEntropy (CE), (from white paper)

We move the gradient computation to the forward function along with an inplace replacement of the logit tensor to avoid them being materialized simultaneously. We also adopt online softmax computation to compute the gradient on the fly. Given the input logits $\bm{x}\in \mathbb{R}^V$, where $V$ is the vocabulary size, and target one-hot encoded label $\bm{t}$, the output probabilities are given as:
\begin{align}
\bm{y} =\textrm{softmax}(\bm{x})
\end{align}
and the cross-entopy loss is defined as $\mathcal{L} = -\sum_i t_i \log (y_i)$. The gradient back-propagated to $\bm{x}$ is given by:
\begin{align}
\nabla_{\bm{x}}\mathcal{L} = \bm{y} - \bm{t}. 
\end{align}
Additionally, we also employ the safe $\log$ operation to avoid numerical instabilities.

#### Profiling

In [2]:
!mkdir -p traces

In [2]:
torch_ce_loss = CrossEntropyLoss().to(device)
cross_entropy_profile(SEQ_LEN, VOCAB_SIZE, torch_ce_loss)

/home/dmasloff/eff-dl/effdl26-liger-kernel-report/.venv/lib/python3.12/site-packages/torch/profiler/profiler.py:217: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(


In [3]:
torch_ce_loss = LigerCrossEntropyLoss().to(device)
cross_entropy_profile(SEQ_LEN, VOCAB_SIZE, torch_ce_loss)

### Part 3. LigerFusedLinearCrossEntropyLoss

In [1]:
import torch
import torch.nn.functional as F
from liger_kernel.transformers import LigerCrossEntropyLoss, LigerFusedLinearCrossEntropyLoss
from torch.nn import CrossEntropyLoss


device = torch.device('cuda')


def liger_fused_linear_cross_enropy_loss_profile(seq_len, vocab_size, hidden_size):
    torch.cuda.empty_cache()
    with torch.profiler.profile(
        activities=[torch.profiler.ProfilerActivity.CPU, torch.profiler.ProfilerActivity.CUDA],
        schedule=torch.profiler.schedule(wait=3, warmup=3, active=1),
        record_shapes=True
    ) as prof:
        loss_fn = LigerFusedLinearCrossEntropyLoss()
        lm_head_weights = torch.randn(vocab_size, hidden_size, device=device, requires_grad=True)
        hiddens = torch.randn(seq_len, hidden_size, device=device, requires_grad=True)
        targets = torch.randint(low=0, high=vocab_size, size=(seq_len,), device=device)

        for _ in range(7):
            loss = loss_fn(lm_head_weights, hiddens, targets)
            loss.backward()

            prof.step()
        
    torch.cuda.synchronize()
    prof.export_chrome_trace(f"./traces/{loss_fn.__repr__()[:-2]}_LMHead.json")


def liger_cross_enropy_loss_profile(seq_len, vocab_size, hidden_size):
    torch.cuda.empty_cache()
    with torch.profiler.profile(
        activities=[torch.profiler.ProfilerActivity.CPU, torch.profiler.ProfilerActivity.CUDA],
        schedule=torch.profiler.schedule(wait=3, warmup=3, active=1),
        record_shapes=True
    ) as prof:
        loss_fn = LigerCrossEntropyLoss()
        lm_head = torch.nn.Linear(hidden_size, vocab_size, bias=False, device=device)
        hiddens = torch.randn(seq_len, hidden_size, device=device, requires_grad=True)
        targets = torch.randint(low=0, high=vocab_size, size=(seq_len,), device=device)

        for _ in range(7):
            logits = lm_head(hiddens)
            loss = loss_fn(logits, targets)
            loss.backward()

            prof.step()
        
    torch.cuda.synchronize()
    prof.export_chrome_trace(f"./traces/{loss_fn.__repr__()[:-2]}_LMHead.json")


def liger_fused_linear_cross_enropy_loss_memory_snapshot(seq_len, vocab_size, hidden_size):
    torch.cuda.empty_cache()
    torch.cuda.memory._record_memory_history()

    loss_fn = LigerFusedLinearCrossEntropyLoss()
    lm_head_weights = torch.randn(vocab_size, hidden_size, device=device, requires_grad=True)
    hiddens = torch.randn(seq_len, hidden_size, device=device, requires_grad=True)
    targets = torch.randint(low=0, high=vocab_size, size=(seq_len,), device=device)

    loss = loss_fn(lm_head_weights, hiddens, targets)
    loss.backward()
        
    torch.cuda.synchronize()
    torch.cuda.memory._dump_snapshot(f"./snapshots/{loss_fn.__repr__()[:-2]}_LMHead.pickle")

def liger_cross_enropy_loss_memory_snapshot(seq_len, vocab_size, hidden_size):
    torch.cuda.empty_cache()
    torch.cuda.memory._record_memory_history()

    loss_fn = LigerCrossEntropyLoss()
    lm_head = torch.nn.Linear(hidden_size, vocab_size, bias=False, device=device)
    hiddens = torch.randn(seq_len, hidden_size, device=device, requires_grad=True)
    targets = torch.randint(low=0, high=vocab_size, size=(seq_len,), device=device)

    logits = lm_head(hiddens)
    loss = loss_fn(logits, targets)
    loss.backward()
    
    torch.cuda.synchronize()
    torch.cuda.memory._dump_snapshot(f"./snapshots/{loss_fn.__repr__()[:-2]}_LMHead.pickle")


In [3]:
liger_fused_linear_cross_enropy_loss_profile(16_000, 128_000, 2048)
liger_cross_enropy_loss_profile(16_000, 128_000, 2048)

/home/dmasloff/eff-dl/effdl26-liger-kernel-report/.venv/lib/python3.12/site-packages/torch/profiler/profiler.py:217: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(


In [2]:
liger_fused_linear_cross_enropy_loss_memory_snapshot(16_000, 128_000, 2048)

In [ ]:
liger_cross_enropy_loss_memory_snapshot(16_000, 128_000, 2048)